# 04 · Knapsack cover inequalities (optional, at home)
**Branch-and-Cut Workshop · UANL & Tecnológico de Monterrey** (Prof. Angélica Salazar, Prof. Vincent Boyer)

*Optional, at home.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you. Cells marked **🤖 AI prompt** contain a prompt you can paste into
> Gemini (the ✨ button in Colab), ChatGPT, Claude… **Always read and test the code the AI gives you.**

In [ ]:
import math
import numpy as np
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

def solve_lp(p, W, cap, cuts=()):
    """LP relaxation of max p x, W x <= cap (one row per knapsack), 0 <= x <= 1,
    plus cuts given as (coef, rhs) meaning coef x <= rhs."""
    rows = [list(r) for r in W] + [list(cf) for cf, _ in cuts]
    rhs = list(cap) + [r for _, r in cuts]
    res = linprog(-np.array(p, float), A_ub=rows, b_ub=rhs, bounds=[(0, 1)] * len(p), method="highs")
    return res.x, -res.fun

def cover_cut(C, n):
    """Cover inequality sum_{j in C} x_j <= |C| - 1 as (coef, rhs)."""
    return [1 if j in C else 0 for j in range(n)], len(C) - 1

p, a, cap = [8, 11, 6, 4], [5, 7, 4, 3], 14
x, z = solve_lp(p, [a], [cap])
print("LP:", np.round(x, 3), "z =", round(z, 3))

## ✍️ Exercise (a) — by hand
Find a cover $C$ ($\sum_{j\in C} a_j > 14$) with $\sum_{j \in C} x^*_j > |C| - 1$, add it, and re-solve.

In [ ]:
C = [0, 1, 2]          # TODO: your cover (0-based indices)
print("cover weight:", sum(a[j] for j in C), "> 14 ?", sum(a[j] for j in C) > cap)
print("violation:", sum(x[j] for j in C), "> |C|-1 =", len(C) - 1)
x2, z2 = solve_lp(p, [a], [cap], cuts=[cover_cut(C, len(p))])
print("new LP:", np.round(x2, 3), "z =", round(z2, 3), "-> floor:", math.floor(z2 + 1e-9))

## ✍️ Exercise (b) — separation by hand
Separation problem:
$$\zeta = \min\Big\{\sum_j (1 - x^*_j) z_j : \sum_j a_j z_j \ge b + 1,\; z \in \{0,1\}^n\Big\},
\qquad \zeta < 1 \iff \text{violated cover}.$$

On paper, for the new LP point $x^* = (1, 1, 0, \tfrac23)$ printed above: write the cost $1 - x^*_j$ of each item,
the weight to reach ($b + 1 = 15$), and solve this small knapsack. Which cover do you get? What is $\zeta$?

Then run the two cells below: `separate_cover` solves the same problem exactly by dynamic programming. Does it agree
with you?

In [ ]:
def separate_cover(a, b, x_star):
    """Exact cover separation by DP over (capped) weight.
    dp[w] = (min cost, items) to reach total weight min(w, b+1)."""
    target = b + 1
    dp = {0: (0.0, [])}
    for j, aj in enumerate(a):
        cost_j = max(0.0, 1.0 - x_star[j])
        for w, (cst, items) in list(dp.items()):
            nw = min(target, w + aj)
            nc = cst + cost_j
            if nw not in dp or nc < dp[nw][0] - 1e-12:
                dp[nw] = (nc, items + [j])
    if target in dp and dp[target][0] < 1 - 1e-6:
        return sorted(dp[target][1])
    return None

In [ ]:
print(separate_cover(a, cap, [1, 1, 0.5, 0]))   # (a): expected [0, 1, 2]
print(separate_cover(a, cap, x2))               # (b): your hand answer? (0-based indices)

## Cutting-plane loop with covers
For each knapsack row, separate a violated cover, turn it into a cut, re-solve; stop when nothing is violated.
`make_cut(row, b, C, x)` turns a cover into an inequality: for now the plain cover inequality.

In [ ]:
def plain_cover(row, b, C, x):
    return cover_cut(C, len(row))

def cover_loop(p, W, cap, make_cut=plain_cover, max_rounds=50, verbose=False):
    cuts = []
    x, z = solve_lp(p, W, cap)
    for r in range(max_rounds):
        new = []
        for row, bb in zip(W, cap):
            C = separate_cover(row, bb, x)
            if C is None:
                continue
            cut = make_cut(row, bb, C, x)
            if sum(cf * xv for cf, xv in zip(cut[0], x)) > cut[1] + 1e-6 and cut not in cuts + new:
                new.append(cut)
        if not new:
            break
        cuts += new
        x, z = solve_lp(p, W, cap, cuts)
        if verbose:
            print(f"round {r+1}: +{len(new)} cuts, LP bound {z:.3f}")
    return x, z, cuts

x, z, cuts = cover_loop(p, [a], [cap], verbose=True)
print("cuts:", cuts, "| bound:", round(z, 3))

## ✍️ Exercise (c) — a bigger instance: measure the gap closed
Gap closed $= \dfrac{z_{LP} - z_{LP+cuts}}{z_{LP} - z_{OPT}}$ (100% = the LP bound reaches the optimum).
The optimum comes from SciPy's MILP solver (HiGHS).

In [ ]:
def random_mkp(n=30, m=1, seed=0, tightness=0.10, lo=10, hi=60):
    rng = np.random.default_rng(seed)
    W = rng.integers(lo, hi, size=(m, n))
    p = (W.sum(axis=0) / m + rng.integers(0, 15, size=n)).astype(int)
    cap = (tightness * W.sum(axis=1)).astype(int)
    return p.tolist(), W.tolist(), cap.tolist()

def solve_opt(p, W, cap):
    res = milp(-np.array(p, float), constraints=LinearConstraint(W, -np.inf, cap),
               integrality=np.ones(len(p)), bounds=Bounds(0, 1))
    return -res.fun

def gap_report(make_cut, label, seeds=range(1, 5)):
    for seed in seeds:
        p_, W_, cap_ = random_mkp(seed=seed)
        z_lp = solve_lp(p_, W_, cap_)[1]
        _, z_cut, cs = cover_loop(p_, W_, cap_, make_cut)
        z_opt = solve_opt(p_, W_, cap_)
        gap = (z_lp - z_cut) / (z_lp - z_opt) if z_lp > z_opt + 1e-9 else 1.0
        print(f"[{label}] seed {seed}: LP {z_lp:7.2f} | +{len(cs):2d} cuts {z_cut:7.2f} | "
              f"OPT {z_opt:5.0f} | gap closed {100*gap:5.1f}%")

gap_report(plain_cover, "plain")

**Write down:** LP bound, bound after covers, optimum, gap closed.

Plain covers close only part of the gap. Two reasons: the cover may not be **minimal**, and items outside $C$
get coefficient 0. **Lifting** fixes both.

## 🚀 If time — lifted covers, written with AI
Sequential up-lifting of a minimal cover $C$: start from $\sum_{C} x_j \le |C|-1$ and, for each $j \notin C$ in turn,
$$\alpha_j = |C| - 1 - \max\Big\{\sum_{i \in \text{current}} \alpha_i x_i \;:\; \sum_{i} a_i x_i \le b - a_j,\; x \in \{0,1\}\Big\}.$$

**🤖 AI prompt:**
```text
Write lift_cover(a, b, C, x_star) for the knapsack row sum a_j x_j <= b and a cover C.
1) Make C minimal: drop items (smallest x_star first) while the remaining weight still exceeds b.
2) Start with alpha_j = 1 for j in C, rhs = |C| - 1.
3) For each j not in C, by decreasing x_star[j]: alpha_j = rhs - (max sum alpha_i x_i over the items
   already in the inequality, s.t. sum a_i x_i <= b - a_j, x binary), solved by a 0-1 knapsack DP.
Return (alpha, rhs). Test: a=[11,6,6,5,5,4,1], b=19, C=[2,3,4,5], x_star=[1,0.9,0,0,0,0,0.5]
must give ([2,1,1,1,1,1,0], 3).
```

In [ ]:
def lift_cover(a, b, C, x_star):
    # TODO: paste / write your lifting routine here; return (alpha, rhs)
    raise NotImplementedError

In [ ]:
print(lift_cover([11, 6, 6, 5, 5, 4, 1], 19, [2, 3, 4, 5], [1, 0.9, 0, 0, 0, 0, 0.5]))  # ([2,1,1,1,1,1,0], 3)
gap_report(lift_cover, "lifted")

**Discuss.** How much more gap do lifted covers close? Why is this the cut family that made 0-1 programming practical
(Crowder, Johnson & Padberg, 1983)?